# Perhitungan Manual LSTM dan Pencocokannya dengan Kode

**Nama:** Tika Altiora Sitanggang
**NIM:** 42324025
**Program Studi:** D-III Teknologi Informasi, Institut Teknologi Del
**Mata Kuliah:** CERTAN

---

## Tentang notebook ini

Notebook ini adalah pasangan dari dokumen teori LSTM (Sesi 1). Di dokumen itu saya menghitung satu langkah LSTM secara manual memakai kalkulator. Di sini perhitungan yang sama dikerjakan lewat kode Python, supaya hasilnya bisa dicocokkan.

Isi notebook:

1. Persiapan library
2. Data dan rumus yang dipakai (sama dengan dokumen teori)
3. Perhitungan Forget Gate
4. Perhitungan Input Gate dan Candidate Cell State
5. Perhitungan Cell State baru
6. Perhitungan Output Gate dan Hidden State
7. Tabel pencocokan hasil manual dengan hasil kode
8. Pembuktian memakai layer `LSTM` asli milik Keras (layer yang sama dengan yang dipakai di praktikum)
9. Hubungan dengan model praktikum `LSTM(100)`
10. Kesimpulan

Notebook bisa dijalankan dari atas ke bawah di Google Colab (Runtime, lalu Run all). Tidak perlu upload dataset apa pun.

## 1. Persiapan Library

Library yang dipakai:

- `numpy` untuk perhitungan angka
- `pandas` untuk menampilkan tabel hasil
- `tensorflow` (Keras) untuk membuat layer LSTM asli

Di Google Colab, ketiganya sudah terpasang, jadi cukup di-import saja.

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import LSTM

print("Versi TensorFlow:", tf.__version__)

Versi TensorFlow: 2.22.0-rc0


## 2. Data dan Rumus yang Digunakan

Angka yang dipakai sama persis dengan Bab 4 di dokumen teori. Angka ini hanya contoh sederhana untuk menunjukkan cara kerja rumus, bukan hasil dari dataset praktikum.

**Nilai awal:**

| Simbol | Arti | Nilai |
|---|---|---|
| xₜ | input saat ini | 0,5 |
| hₜ₋₁ | hidden state sebelumnya | 0,1 |
| Cₜ₋₁ | cell state sebelumnya | 0,2 |
| b | bias pada semua gate | 0 |

**Rumus umum LSTM:**

- Forget gate: fₜ = σ(Wf · [hₜ₋₁, xₜ] + bf)
- Input gate: iₜ = σ(Wi · [hₜ₋₁, xₜ] + bi)
- Candidate: C̃ₜ = tanh(WC · [hₜ₋₁, xₜ] + bC)
- Cell state: Cₜ = fₜ × Cₜ₋₁ + iₜ × C̃ₜ
- Output gate: oₜ = σ(Wo · [hₜ₋₁, xₜ] + bo)
- Hidden state: hₜ = oₜ × tanh(Cₜ)

**Bobot yang dipilih** (supaya hitungannya sederhana, sama seperti di dokumen teori):

| Gate | Bobot untuk hₜ₋₁ | Bobot untuk xₜ | Rumus jadi |
|---|---|---|---|
| Forget | 1 | 1 | σ(hₜ₋₁ + xₜ) |
| Input | 0,5 | 0,5 | σ(0,5·hₜ₋₁ + 0,5·xₜ) |
| Candidate | 1 | 1 | tanh(hₜ₋₁ + xₜ) |
| Output | 0,2 | 0,8 | σ(0,2·hₜ₋₁ + 0,8·xₜ) |

Kode di bawah menyimpan semua nilai awal dan bobot tersebut ke dalam variabel.

In [2]:
# Nilai awal
x_t    = 0.5   # input saat ini
h_prev = 0.1   # hidden state sebelumnya
c_prev = 0.2   # cell state sebelumnya

# Bobot: (bobot untuk h_prev, bobot untuk x_t)
Wf_h, Wf_x = 1.0, 1.0    # forget gate
Wi_h, Wi_x = 0.5, 0.5    # input gate
Wc_h, Wc_x = 1.0, 1.0    # candidate cell state
Wo_h, Wo_x = 0.2, 0.8    # output gate

# Bias semuanya 0
bf = bi = bc = bo = 0.0

print("x_t    =", x_t)
print("h_prev =", h_prev)
print("c_prev =", c_prev)

x_t    = 0.5
h_prev = 0.1
c_prev = 0.2


### Fungsi sigmoid dan tanh

Dua fungsi aktivasi yang dipakai LSTM:

- Sigmoid: σ(z) = 1 / (1 + e⁻ᶻ), hasilnya antara 0 sampai 1
- Tanh: tanh(z) = (eᶻ − e⁻ᶻ) / (eᶻ + e⁻ᶻ), hasilnya antara -1 sampai 1

Di kode, rumusnya langsung ditulis sebagai fungsi. Isinya sama dengan rumus yang dihitung di kalkulator.

In [3]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def tanh(z):
    return (np.exp(z) - np.exp(-z)) / (np.exp(z) + np.exp(-z))

# Cek singkat: hasilnya harus sama dengan fungsi bawaan numpy
print("tanh(0.6) rumus  :", tanh(0.6))
print("tanh(0.6) numpy  :", np.tanh(0.6))

tanh(0.6) rumus  : 0.5370495669980353
tanh(0.6) numpy  : 0.5370495669980352


## 3. Forget Gate

Forget gate menentukan seberapa banyak cell state lama yang masih dipertahankan. Nilainya antara 0 sampai 1. Makin dekat ke 1, makin banyak informasi lama yang disimpan.

**Rumus:**

fₜ = σ(Wf · [hₜ₋₁, xₜ] + bf)

Dengan bobot yang dipilih dan bias 0:

fₜ = σ(hₜ₋₁ + xₜ) = σ(0,1 + 0,5) = σ(0,6) = 1 / (1 + e⁻⁰·⁶) ≈ **0,6457**

Artinya sekitar 64,57% informasi dari cell state sebelumnya dipertahankan.

**Hubungan dengan kode:** baris `f_t = sigmoid(...)` di bawah adalah penulisan rumus forget gate dalam Python. Bagian `Wf_h * h_prev + Wf_x * x_t + bf` adalah `Wf · [hₜ₋₁, xₜ] + bf` yang dijabarkan satu per satu.

In [4]:
# Langkah 1: hitung nilai di dalam sigmoid (z)
z_f = Wf_h * h_prev + Wf_x * x_t + bf
print("z_f =", z_f)

# Langkah 2: masukkan ke sigmoid
f_t = sigmoid(z_f)
print("f_t =", round(f_t, 4))

z_f = 0.6
f_t = 0.6457


## 4. Input Gate dan Candidate Cell State

Bagian ini menentukan informasi baru yang akan dimasukkan ke memori. Ada dua hitungan:

- **Input gate (iₜ)** menentukan seberapa besar informasi baru boleh masuk (0 sampai 1).
- **Candidate cell state (C̃ₜ)** adalah calon informasi barunya (-1 sampai 1).

**Rumus:**

iₜ = σ(Wi · [hₜ₋₁, xₜ] + bi)
C̃ₜ = tanh(WC · [hₜ₋₁, xₜ] + bC)

**Perhitungan input gate:**

iₜ = σ(0,5·hₜ₋₁ + 0,5·xₜ) = σ(0,5(0,1) + 0,5(0,5)) = σ(0,05 + 0,25) = σ(0,3) ≈ **0,5744**

**Perhitungan candidate:**

C̃ₜ = tanh(hₜ₋₁ + xₜ) = tanh(0,1 + 0,5) = tanh(0,6) ≈ **0,5370**

**Hubungan dengan kode:** `i_t` memakai `sigmoid`, sedangkan `c_tilde` memakai `tanh`. Pemilihan fungsinya sama dengan rumus: sigmoid untuk gate, tanh untuk candidate.

In [5]:
# --- Input gate ---
z_i = Wi_h * h_prev + Wi_x * x_t + bi
i_t = sigmoid(z_i)
print("z_i     =", z_i)
print("i_t     =", round(i_t, 4))

# --- Candidate cell state ---
z_c = Wc_h * h_prev + Wc_x * x_t + bc
c_tilde = tanh(z_c)
print("z_c     =", z_c)
print("c_tilde =", round(c_tilde, 4))

z_i     = 0.3
i_t     = 0.5744
z_c     = 0.6
c_tilde = 0.537


## 5. Cell State Baru

Cell state baru adalah gabungan dari dua hal:

- informasi lama yang masih dipertahankan: fₜ × Cₜ₋₁
- informasi baru yang sudah disaring input gate: iₜ × C̃ₜ

**Rumus:**

Cₜ = fₜ × Cₜ₋₁ + iₜ × C̃ₜ

**Perhitungan:**

Cₜ = (0,6457 × 0,2) + (0,5744 × 0,5370) = 0,1291 + 0,3085 ≈ **0,4376**

**Hubungan dengan kode:** baris `c_t = f_t * c_prev + i_t * c_tilde` ditulis sama persis dengan rumusnya. Dua bagian penjumlahannya saya pisah dulu supaya bisa dicocokkan dengan angka 0,1291 dan 0,3085 di perhitungan manual.

In [6]:
bagian_lama = f_t * c_prev        # informasi lama yang dipertahankan
bagian_baru = i_t * c_tilde       # informasi baru yang dimasukkan

c_t = bagian_lama + bagian_baru

print("f_t * c_prev   =", round(bagian_lama, 4))
print("i_t * c_tilde  =", round(bagian_baru, 4))
print("c_t            =", round(c_t, 4))

f_t * c_prev   = 0.1291
i_t * c_tilde  = 0.3085
c_t            = 0.4376


## 6. Output Gate dan Hidden State

Output gate menentukan bagian dari cell state yang dikeluarkan sebagai hidden state. Hidden state inilah yang diteruskan ke langkah waktu berikutnya atau dipakai untuk prediksi.

**Rumus:**

oₜ = σ(Wo · [hₜ₋₁, xₜ] + bo)
hₜ = oₜ × tanh(Cₜ)

**Perhitungan output gate:**

oₜ = σ(0,2·hₜ₋₁ + 0,8·xₜ) = σ(0,2(0,1) + 0,8(0,5)) = σ(0,02 + 0,4) = σ(0,42) ≈ **0,6035**

**Perhitungan hidden state:**

hₜ = 0,6035 × tanh(0,4376) = 0,6035 × 0,4117 ≈ **0,2484**

**Hubungan dengan kode:** `o_t` dihitung dengan sigmoid, lalu `h_t = o_t * tanh(c_t)` memakai cell state baru yang sudah dihitung di bagian 5.

In [7]:
# --- Output gate ---
z_o = Wo_h * h_prev + Wo_x * x_t + bo
o_t = sigmoid(z_o)
print("z_o          =", z_o)
print("o_t          =", round(o_t, 4))

# --- Hidden state ---
tanh_ct = tanh(c_t)
h_t = o_t * tanh_ct
print("tanh(c_t)    =", round(tanh_ct, 4))
print("h_t          =", round(h_t, 4))

z_o          = 0.42000000000000004
o_t          = 0.6035
tanh(c_t)    = 0.4117
h_t          = 0.2484


## 7. Pencocokan Hasil Perhitungan Manual dengan Hasil Kode

Kolom "Manual" diambil dari angka akhir di dokumen teori (Bab 5), yaitu angka yang saya hitung dengan kalkulator dan dibulatkan 4 angka di belakang koma. Kolom "Kode" adalah hasil dari sel-sel di atas.

Kalau selisihnya kurang dari 0,0001, berarti hasilnya cocok (selisih kecil itu hanya karena pembulatan di perhitungan manual).

In [8]:
manual = {
    "f_t (forget gate)":    0.6457,
    "i_t (input gate)":     0.5744,
    "C~_t (candidate)":     0.5370,
    "C_t (cell state)":     0.4376,
    "o_t (output gate)":    0.6035,
    "h_t (hidden state)":   0.2484,
}
kode = {
    "f_t (forget gate)":    f_t,
    "i_t (input gate)":     i_t,
    "C~_t (candidate)":     c_tilde,
    "C_t (cell state)":     c_t,
    "o_t (output gate)":    o_t,
    "h_t (hidden state)":   h_t,
}

tabel = pd.DataFrame({
    "Manual": list(manual.values()),
    "Kode":   [round(v, 4) for v in kode.values()],
}, index=list(manual.keys()))
tabel["Selisih"] = (tabel["Manual"] - [kode[k] for k in manual]).abs().round(6)
tabel["Cocok?"] = tabel["Selisih"] < 0.0001
tabel

,Manual,Kode,Selisih,Cocok?
f_t (forget gate),0.6457,0.6457,0.000044,True
i_t (input gate),0.5744,0.5744,0.000043,True
C~_t (candidate),0.5370,0.5370,0.000050,True
C_t (cell state),0.4376,0.4376,0.000035,True
o_t (output gate),0.6035,0.6035,0.000017,True
h_t (hidden state),0.2484,0.2484,0.000043,True


In [9]:
# Pengecekan otomatis: kalau ada yang tidak cocok, akan muncul error
for nama in manual:
    assert abs(manual[nama] - kode[nama]) < 0.0001, f"{nama} tidak cocok"

print("Semua nilai perhitungan manual cocok dengan hasil kode.")

Semua nilai perhitungan manual cocok dengan hasil kode.


## 8. Pembuktian dengan Layer LSTM Asli dari Keras

Sampai bagian 7, saya baru mencocokkan hitungan manual dengan kode Python yang saya tulis sendiri. Bagian ini lebih kuat: saya memakai layer `LSTM` dari Keras, yaitu layer yang sama dengan yang dipakai di praktikum, lalu mengisi bobotnya dengan angka contoh di atas. Kalau layer asli itu menghasilkan hₜ dan Cₜ yang sama dengan perhitungan manual, berarti rumus yang saya hitung memang yang dikerjakan di dalam layer LSTM.

Beberapa hal yang perlu diketahui supaya hasilnya bisa sama:

- Layer dibuat dengan **1 unit** (`LSTM(1)`), karena perhitungan manual hanya untuk satu unit.
- Fungsi aktivasi memakai bawaan Keras: **tanh** untuk candidate dan hidden state, **sigmoid** untuk gate. Ini sesuai rumus di dokumen teori. (Pada model praktikum ada `activation='relu'`, tapi untuk pembuktian rumus ini dipakai tanh sesuai teori.)
- Keras menyimpan bobot dalam 3 bagian: `kernel` (bobot untuk xₜ), `recurrent_kernel` (bobot untuk hₜ₋₁), dan `bias`.
- Urutan gate di dalam Keras adalah **input, forget, candidate, output**. Jadi nilai bobot harus ditulis berurutan seperti itu.
- hₜ₋₁ dan Cₜ₋₁ dimasukkan lewat `initial_state`.

Susunan bobot yang diisikan:

| Urutan di Keras | Gate | kernel (untuk xₜ) | recurrent_kernel (untuk hₜ₋₁) | bias |
|---|---|---|---|---|
| 1 | Input | 0,5 | 0,5 | 0 |
| 2 | Forget | 1,0 | 1,0 | 0 |
| 3 | Candidate | 1,0 | 1,0 | 0 |
| 4 | Output | 0,8 | 0,2 | 0 |

In [10]:
# Buat layer LSTM dengan 1 unit. return_state=True supaya h dan c ikut dikeluarkan.
lstm_layer = LSTM(1, return_state=True)

# Data input berbentuk (batch, time step, fitur) = (1, 1, 1)
x_input = np.array([[[x_t]]], dtype="float32")
h0 = np.array([[h_prev]], dtype="float32")
c0 = np.array([[c_prev]], dtype="float32")

# Panggil sekali dulu supaya layer terbentuk (bobotnya dibuat)
_ = lstm_layer(x_input, initial_state=[h0, c0])

# Isi bobot sesuai tabel di atas (urutan: input, forget, candidate, output)
kernel           = np.array([[Wi_x, Wf_x, Wc_x, Wo_x]], dtype="float32")   # untuk x_t
recurrent_kernel = np.array([[Wi_h, Wf_h, Wc_h, Wo_h]], dtype="float32")   # untuk h_prev
bias             = np.array([bi, bf, bc, bo], dtype="float32")

lstm_layer.set_weights([kernel, recurrent_kernel, bias])

# Jalankan lagi dengan bobot yang sudah diisi
output, h_keras, c_keras = lstm_layer(x_input, initial_state=[h0, c0])

h_keras = float(h_keras.numpy()[0, 0])
c_keras = float(c_keras.numpy()[0, 0])

print("Hasil layer LSTM Keras:")
print("  h_t =", round(h_keras, 4))
print("  C_t =", round(c_keras, 4))

Hasil layer LSTM Keras:
  h_t = 0.2484
  C_t = 0.4376


In [11]:
hasil = pd.DataFrame({
    "Manual (dokumen teori)": [0.4376, 0.2484],
    "Kode Python (bagian 3-6)": [round(c_t, 4), round(h_t, 4)],
    "Layer LSTM Keras": [round(c_keras, 4), round(h_keras, 4)],
}, index=["C_t (cell state)", "h_t (hidden state)"])
hasil

,Manual (dokumen teori),Kode Python (bagian 3-6),Layer LSTM Keras
C_t (cell state),0.4376,0.4376,0.4376
h_t (hidden state),0.2484,0.2484,0.2484


In [12]:
assert abs(h_keras - 0.2484) < 0.0001
assert abs(c_keras - 0.4376) < 0.0001
print("Layer LSTM Keras menghasilkan h_t dan C_t yang sama dengan perhitungan manual.")

Layer LSTM Keras menghasilkan h_t dan C_t yang sama dengan perhitungan manual.


### Penjelasan hasil

Tiga cara menghitung (kalkulator, kode Python sendiri, dan layer LSTM Keras) memberikan hasil yang sama: Cₜ ≈ 0,4376 dan hₜ ≈ 0,2484. Ini menunjukkan bahwa di dalam layer `LSTM` pada praktikum, yang terjadi memang urutan yang sama seperti yang saya hitung manual: forget gate, input gate, cell state, lalu output gate.

## 9. Hubungan dengan Model `LSTM(100)` pada Praktikum

Di praktikum, model dibuat seperti ini:

```python
model_lstm = Sequential()
model_lstm.add(LSTM(100, activation='relu', input_shape=(length, n_features)))
model_lstm.add(Dropout(0.2))
model_lstm.add(Dense(1))
```

Perbedaannya dengan contoh manual:

| | Contoh manual | Model praktikum |
|---|---|---|
| Jumlah unit | 1 | 100 |
| Bobot | ditentukan sendiri | dipelajari dari data saat training |
| Panjang input | 1 time step | 12 time step (`length = 12`) |
| Fitur | 1 | 1 (`n_features = 1`) |

Karena itu hasil manual tidak harus sama dengan prediksi model praktikum. Prosesnya sama, tetapi di praktikum ada 100 unit dan bobotnya diatur otomatis oleh training.

Sel di bawah membuat ulang susunan model praktikum (tanpa training) untuk menunjukkan jumlah parameternya.

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

length = 12
n_features = 1

model_lstm = Sequential()
model_lstm.add(Input(shape=(length, n_features)))
model_lstm.add(LSTM(100, activation='relu'))
model_lstm.add(Dropout(0.2))
model_lstm.add(Dense(1))

model_lstm.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 100)            │        40,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,901 (159.77 KB)

 Trainable params: 40,901 (159.77 KB)

 Non-trainable params: 0 (0.00 B)

### Dari mana angka 40.800 pada layer LSTM?

Setiap unit LSTM punya 4 bagian hitungan (forget, input, candidate, output). Tiap bagian punya bobot untuk xₜ, bobot untuk hₜ₋₁, dan satu bias. Jadi rumusnya:

parameter = 4 × (unit × (fitur + unit) + unit)

Dengan 100 unit dan 1 fitur:

4 × (100 × (1 + 100) + 100) = 4 × 10.200 = **40.800**

Angka ini sama dengan yang tampil di slide praktikum (`lstm_1`: 40.800). Ditambah `Dense(1)` yang punya 101 parameter (100 bobot + 1 bias), totalnya 40.901.

In [14]:
unit = 100
fitur = 1

param_lstm  = 4 * (unit * (fitur + unit) + unit)
param_dense = unit * 1 + 1

print("Parameter layer LSTM  :", param_lstm)
print("Parameter layer Dense :", param_dense)
print("Total                 :", param_lstm + param_dense)

Parameter layer LSTM  : 40800
Parameter layer Dense : 101
Total                 : 40901


## 10. Kesimpulan

- Satu langkah LSTM terdiri dari empat bagian: forget gate, input gate (bersama candidate), pembaruan cell state, dan output gate.
- Dengan nilai xₜ = 0,5, hₜ₋₁ = 0,1, dan Cₜ₋₁ = 0,2, hasilnya: fₜ ≈ 0,6457, iₜ ≈ 0,5744, C̃ₜ ≈ 0,5370, Cₜ ≈ 0,4376, oₜ ≈ 0,6035, dan hₜ ≈ 0,2484.
- Hasil kode Python sama dengan hasil perhitungan manual di dokumen teori.
- Layer `LSTM` dari Keras, yang diisi bobot contoh yang sama, juga menghasilkan Cₜ dan hₜ yang sama. Jadi rumus yang dihitung manual memang yang dikerjakan layer LSTM di praktikum.
- Pada model praktikum `LSTM(100)`, proses yang sama berjalan pada 100 unit dengan bobot hasil training, sehingga angkanya tidak harus sama dengan contoh manual.